<a href="https://colab.research.google.com/github/DominicBuxton/Hallou_Cellpose/blob/main/Segmentation_stitching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Ctrl stitch

In [ ]:
import numpy as np
from skimage.io import imread, imsave
import pandas as pd
from skimage.segmentation import relabel_sequential
import tifffile

def relabel_with_offset(mask, offset):
    """
    Shift all non-zero labels in a mask by a given offset.
    Background (0) is preserved.
    """
    mask32 = mask.astype(np.uint32, copy=False) #Ensure no overflow by converting to int32 before math
    relabeled = np.where(mask32 > 0, mask32 + np.uint32(offset), 0)
    return relabeled


def overlay_submasks(image_dimensions, relabeled_tiles, submask_locations):
    """
    Overlay relabelled tiles onto a blank canvas at specified (x, y) positions.

    Parameters
    ----------
    image_dimensions : tuple
        Shape of the final canvas (height, width), matching large_image.shape.
    relabeled_tiles : list of np.ndarray
        List of 2D relabelled segmentation masks.
    submask_locations : list of [x, y]
        Top-left (x=col, y=row) position for each tile on the canvas.

    Returns
    -------
    final_mask : np.ndarray
        Canvas with all tiles overlaid, dtype uint32.
    """
    final_mask = np.zeros(image_dimensions[:2], dtype=np.uint32)

    for tile, (x, y) in zip(relabeled_tiles, submask_locations):
        tile_h, tile_w = tile.shape[:2]

        # Compute canvas region this tile maps to
        y_end = y + tile_h
        x_end = x + tile_w

        # Clip to canvas bounds (handles edge tiles that may exceed canvas)
        y_end_clipped = min(y_end, image_dimensions[0])
        x_end_clipped = min(x_end, image_dimensions[1])

        # Corresponding slice of the tile (in case it was clipped)
        tile_y_end = tile_h - (y_end - y_end_clipped)
        tile_x_end = tile_w - (x_end - x_end_clipped)

        canvas_region = final_mask[y:y_end_clipped, x:x_end_clipped]
        tile_region   = tile[:tile_y_end, :tile_x_end]

        # Only write tile pixels where the canvas is still background (0)
        # This avoids overwriting cells at overlapping tile borders
        background = canvas_region == 0
        canvas_region[background] = tile_region[background]

        final_mask[y:y_end_clipped, x:x_end_clipped] = canvas_region

    return final_mask


# --- Main pipeline ---

# Load locations from CSV (expects columns 'x' and 'y')
locations_df = pd.read_csv("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/FMCtrl_BBox_values.csv", dtype = int)
submask_locations = locations_df[["BX", "BY"]].values.tolist()


# Load tiles (must be in the same order as submask_locations)

tiles = [imread("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/Ctrl_F1/nuclear_labels.tif"),
         imread("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/Ctrl_F2/nuclear_labels.tif"),
         imread("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/Ctrl_F3/nuclear_labels.tif"),
         imread("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/Ctrl_M1/nuclear_labels.tif"),
         imread("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/Ctrl_M2/nuclear_labels.tif"),
         imread("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/Ctrl_M3/nuclear_labels.tif")
         ]

# Relabel tiles with unique IDs

offset = 0
relabeled_tiles = []

for tile in tiles:
    max_label = tile.max()
    if max_label > 0:
        relabeled = relabel_with_offset(tile, offset)
        offset += relabeled.max()          # next tile starts above this one's range
    else:
        relabeled = tile             # empty tile, no cells
    relabeled_tiles.append(relabeled)

# Load large image just to get canvas dimensions
large_image_shape = (13706, 51100) #expects y, x format

# Build final mask
print("overlaying submasks")
print(f"large image shape is {large_image_shape}")
print(f"submask locations are {submask_locations}")
final_mask = overlay_submasks(large_image_shape, relabeled_tiles, submask_locations)
#compacting to avoid gaps, ensure mask IDs are contiguous and sequential
final_mask_compacted, fw, inv = relabel_sequential(final_mask)

# Save
tifffile.imwrite(
    "/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/Ctrl_nuclear_labels.tif",
    final_mask_compacted.astype(np.uint32),
    compression=None  # or 'zlib' if file size is a concern
)

#checklist = csv file, submask files, large image size, final filename

overlaying submasks
large image shape is (13706, 51100)
submask locations are [[2568, 9864], [6096, 8376], [4186, 6506], [28188, 8538], [26940, 6602], [27660, 1658]]


# D3 stitch

In [ ]:
import numpy as np
from skimage.io import imread, imsave
import pandas as pd
from skimage.segmentation import relabel_sequential
import tifffile

def relabel_with_offset(mask, offset):
    """
    Shift all non-zero labels in a mask by a given offset.
    Background (0) is preserved.
    """
    mask32 = mask.astype(np.uint32, copy=False) #Ensure no overflow by converting to int32 before math
    relabeled = np.where(mask32 > 0, mask32 + np.uint32(offset), 0)
    return relabeled


def overlay_submasks(image_dimensions, relabeled_tiles, submask_locations):
    """
    Overlay relabelled tiles onto a blank canvas at specified (x, y) positions.

    Parameters
    ----------
    image_dimensions : tuple
        Shape of the final canvas (height, width), matching large_image.shape.
    relabeled_tiles : list of np.ndarray
        List of 2D relabelled segmentation masks.
    submask_locations : list of [x, y]
        Top-left (x=col, y=row) position for each tile on the canvas.

    Returns
    -------
    final_mask : np.ndarray
        Canvas with all tiles overlaid, dtype uint32.
    """
    final_mask = np.zeros(image_dimensions[:2], dtype=np.uint32)

    for tile, (x, y) in zip(relabeled_tiles, submask_locations):
        tile_h, tile_w = tile.shape[:2]

        # Compute canvas region this tile maps to
        y_end = y + tile_h
        x_end = x + tile_w

        # Clip to canvas bounds (handles edge tiles that may exceed canvas)
        y_end_clipped = min(y_end, image_dimensions[0])
        x_end_clipped = min(x_end, image_dimensions[1])

        # Corresponding slice of the tile (in case it was clipped)
        tile_y_end = tile_h - (y_end - y_end_clipped)
        tile_x_end = tile_w - (x_end - x_end_clipped)

        canvas_region = final_mask[y:y_end_clipped, x:x_end_clipped]
        tile_region   = tile[:tile_y_end, :tile_x_end]

        # Only write tile pixels where the canvas is still background (0)
        # This avoids overwriting cells at overlapping tile borders
        background = canvas_region == 0
        canvas_region[background] = tile_region[background]

        final_mask[y:y_end_clipped, x:x_end_clipped] = canvas_region

    return final_mask


# --- Main pipeline ---

# Load locations from CSV (expects columns 'x' and 'y')
locations_df = pd.read_csv("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/FMD3_BBox_values.csv", dtype = int)
submask_locations = locations_df[["BX", "BY"]].values.tolist()


# Load tiles (must be in the same order as submask_locations)

tiles = [imread("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/D3_F1/nuclear_labels.tif"),
         imread("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/D3_F2/nuclear_labels.tif"),
         imread("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/D3_F3/nuclear_labels.tif"),
         imread("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/D3_F4/nuclear_labels.tif"),
         imread("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/D3_M1/nuclear_labels.tif"),
         imread("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/D3_M2/nuclear_labels.tif"),
         imread("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/D3_M3/nuclear_labels.tif"),
         imread("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/D3_M4/nuclear_labels.tif")
         ]

# Relabel tiles with unique IDs

offset = 0
relabeled_tiles = []

for tile in tiles:
    max_label = tile.max()
    if max_label > 0:
        relabeled = relabel_with_offset(tile, offset)
        offset += relabeled.max()          # next tile starts above this one's range
    else:
        relabeled = tile             # empty tile, no cells
    relabeled_tiles.append(relabeled)

# Load large image just to get canvas dimensions
large_image_shape = (23878, 51109) #expects y, x format

# Build final mask
print("overlaying submasks")
print(f"large image shape is {large_image_shape}")
print(f"submask locations are {submask_locations}")
final_mask = overlay_submasks(large_image_shape, relabeled_tiles, submask_locations)
#compacting to avoid gaps, ensure mask IDs are contiguous and sequential
final_mask_compacted, fw, inv = relabel_sequential(final_mask)

# Save
tifffile.imwrite(
    "/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/D3_nuclear_labels.tif",
    final_mask_compacted.astype(np.uint32),
    compression=None  # or 'zlib' if file size is a concern
)

#checklist = csv file, submask files, large image size, final filename

overlaying submasks
large image shape is (23878, 51109)
submask locations are [[4224, 12288], [3456, 10240], [1920, 7808], [1408, 4992], [26624, 14720], [28672, 10496], [28544, 6400], [28416, 2048]]


# D7 stitch

In [ ]:
import numpy as np
from skimage.io import imread, imsave
import pandas as pd
from skimage.segmentation import relabel_sequential
import tifffile

def relabel_with_offset(mask, offset):
    """
    Shift all non-zero labels in a mask by a given offset.
    Background (0) is preserved.
    """
    mask32 = mask.astype(np.uint32, copy=False) #Ensure no overflow by converting to int32 before math
    relabeled = np.where(mask32 > 0, mask32 + np.uint32(offset), 0)
    return relabeled


def overlay_submasks(image_dimensions, relabeled_tiles, submask_locations):
    """
    Overlay relabelled tiles onto a blank canvas at specified (x, y) positions.

    Parameters
    ----------
    image_dimensions : tuple
        Shape of the final canvas (height, width), matching large_image.shape.
    relabeled_tiles : list of np.ndarray
        List of 2D relabelled segmentation masks.
    submask_locations : list of [x, y]
        Top-left (x=col, y=row) position for each tile on the canvas.

    Returns
    -------
    final_mask : np.ndarray
        Canvas with all tiles overlaid, dtype uint32.
    """
    final_mask = np.zeros(image_dimensions[:2], dtype=np.uint32)

    for tile, (x, y) in zip(relabeled_tiles, submask_locations):
        tile_h, tile_w = tile.shape[:2]

        # Compute canvas region this tile maps to
        y_end = y + tile_h
        x_end = x + tile_w

        # Clip to canvas bounds (handles edge tiles that may exceed canvas)
        y_end_clipped = min(y_end, image_dimensions[0])
        x_end_clipped = min(x_end, image_dimensions[1])

        # Corresponding slice of the tile (in case it was clipped)
        tile_y_end = tile_h - (y_end - y_end_clipped)
        tile_x_end = tile_w - (x_end - x_end_clipped)

        canvas_region = final_mask[y:y_end_clipped, x:x_end_clipped]
        tile_region   = tile[:tile_y_end, :tile_x_end]

        # Only write tile pixels where the canvas is still background (0)
        # This avoids overwriting cells at overlapping tile borders
        background = canvas_region == 0
        canvas_region[background] = tile_region[background]

        final_mask[y:y_end_clipped, x:x_end_clipped] = canvas_region

    return final_mask


# --- Main pipeline ---

# Load locations from CSV (expects columns 'x' and 'y')
locations_df = pd.read_csv("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/FMD7_BBox_values.csv", dtype = int)
submask_locations = locations_df[["BX", "BY"]].values.tolist()


# Load tiles (must be in the same order as submask_locations)

tiles = [imread("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/D7_F1/nuclear_labels.tif"),
         imread("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/D7_F2/nuclear_labels.tif"),
         imread("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/D7_F3/nuclear_labels.tif"),
         imread("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/D7_F4/nuclear_labels.tif"),
         imread("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/D7_M1/nuclear_labels.tif"),
         imread("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/D7_M2/nuclear_labels.tif"),
         imread("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/D7_M3/nuclear_labels.tif"),
         imread("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/D7_M4/nuclear_labels.tif")
         ]

# Relabel tiles with unique IDs

offset = 0
relabeled_tiles = []

for tile in tiles:
    max_label = tile.max()
    if max_label > 0:
        relabeled = relabel_with_offset(tile, offset)
        offset += relabeled.max()          # next tile starts above this one's range
    else:
        relabeled = tile             # empty tile, no cells
    relabeled_tiles.append(relabeled)

# Load large image just to get canvas dimensions
large_image_shape = (20483, 53936) #expects y, x format

# Build final mask
print("overlaying submasks")
print(f"large image shape is {large_image_shape}")
print(f"submask locations are {submask_locations}")
final_mask = overlay_submasks(large_image_shape, relabeled_tiles, submask_locations)
#compacting to avoid gaps, ensure mask IDs are contiguous and sequential
final_mask_compacted, fw, inv = relabel_sequential(final_mask)

# Save
tifffile.imwrite(
    "/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/D7_nuclear_labels.tif",
    final_mask_compacted.astype(np.uint32),
    compression=None  # or 'zlib' if file size is a concern
)

#checklist = csv file, submask files, large image size, final filename

overlaying submasks
large image shape is (20483, 53936)
submask locations are [[6016, 10880], [4096, 11776], [5120, 7680], [2304, 3328], [28800, 10624], [26496, 6016], [27264, 3584], [25600, 896]]


# D10 Stitch

In [ ]:
import numpy as np
from skimage.io import imread, imsave
import pandas as pd
from skimage.segmentation import relabel_sequential
import tifffile

def relabel_with_offset(mask, offset):
    """
    Shift all non-zero labels in a mask by a given offset.
    Background (0) is preserved.
    """
    mask32 = mask.astype(np.uint32, copy=False) #Ensure no overflow by converting to int32 before math
    relabeled = np.where(mask32 > 0, mask32 + np.uint32(offset), 0)
    return relabeled


def overlay_submasks(image_dimensions, relabeled_tiles, submask_locations):
    """
    Overlay relabelled tiles onto a blank canvas at specified (x, y) positions.

    Parameters
    ----------
    image_dimensions : tuple
        Shape of the final canvas (height, width), matching large_image.shape.
    relabeled_tiles : list of np.ndarray
        List of 2D relabelled segmentation masks.
    submask_locations : list of [x, y]
        Top-left (x=col, y=row) position for each tile on the canvas.

    Returns
    -------
    final_mask : np.ndarray
        Canvas with all tiles overlaid, dtype uint32.
    """
    final_mask = np.zeros(image_dimensions[:2], dtype=np.uint32)

    for tile, (x, y) in zip(relabeled_tiles, submask_locations):
        tile_h, tile_w = tile.shape[:2]

        # Compute canvas region this tile maps to
        y_end = y + tile_h
        x_end = x + tile_w

        # Clip to canvas bounds (handles edge tiles that may exceed canvas)
        y_end_clipped = min(y_end, image_dimensions[0])
        x_end_clipped = min(x_end, image_dimensions[1])

        # Corresponding slice of the tile (in case it was clipped)
        tile_y_end = tile_h - (y_end - y_end_clipped)
        tile_x_end = tile_w - (x_end - x_end_clipped)

        canvas_region = final_mask[y:y_end_clipped, x:x_end_clipped]
        tile_region   = tile[:tile_y_end, :tile_x_end]

        # Only write tile pixels where the canvas is still background (0)
        # This avoids overwriting cells at overlapping tile borders
        background = canvas_region == 0
        canvas_region[background] = tile_region[background]

        final_mask[y:y_end_clipped, x:x_end_clipped] = canvas_region

    return final_mask


# --- Main pipeline ---

# Load locations from CSV (expects columns 'x' and 'y')
locations_df = pd.read_csv("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/FMD10_BBox_values.csv", dtype = int)
submask_locations = locations_df[["BX", "BY"]].values.tolist()


# Load tiles (must be in the same order as submask_locations)

tiles = [imread("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/D10_F1/nuclear_labels.tif"),
         imread("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/D10_F2/nuclear_labels.tif"),
         imread("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/D10_F3/nuclear_labels.tif"),
         imread("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/D10_F4/nuclear_labels.tif"),
         imread("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/D10_M1/nuclear_labels.tif"),
         imread("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/D10_M2/nuclear_labels.tif"),
         imread("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/D10_M3/nuclear_labels.tif"),
         imread("/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/D10_M4/nuclear_labels.tif")
         ]

# Relabel tiles with unique IDs

offset = 0
relabeled_tiles = []

for tile in tiles:
    max_label = tile.max()
    if max_label > 0:
        relabeled = relabel_with_offset(tile, offset)
        offset += relabeled.max()          # next tile starts above this one's range
    else:
        relabeled = tile             # empty tile, no cells
    relabeled_tiles.append(relabeled)

# Load large image just to get canvas dimensions
large_image_shape = (23869, 53940) #expects y, x format

# Build final mask
print("overlaying submasks")
print(f"large image shape is {large_image_shape}")
print(f"submask locations are {submask_locations}")
final_mask = overlay_submasks(large_image_shape, relabeled_tiles, submask_locations)
#compacting to avoid gaps, ensure mask IDs are contiguous and sequential
final_mask_compacted, fw, inv = relabel_sequential(final_mask)

# Save
tifffile.imwrite(
    "/content/drive/MyDrive/xen_dapi_images/xen_dapi to google drive/Registered Images - dapi only/D10_nuclear_labels.tif",
    final_mask_compacted.astype(np.uint32),
    compression=None  # or 'zlib' if file size is a concern
)

#checklist = csv file, submask files, large image size, final filename

overlaying submasks
large image shape is (23869, 53940)
submask locations are [[4736, 18304], [4352, 14336], [2944, 10496], [2560, 8064], [27520, 16128], [27264, 12800], [28416, 9472], [29184, 3072]]
